# ISIC 2024 - Patient-Level Data Splitting

This notebook creates a reliable data splitting pipeline for training and evaluating skin cancer classification models on the ISIC 2024 dataset.

The splitting strategy is designed specifically for medical imaging data, where multiple samples may belong to the same patient.

---

## Objectives

- Prevent patient-level data leakage
- Preserve class distribution using stratification
- Create Train / Validation / Holdout splits
- Ensure reproducibility for experiments

---

## Key Considerations

- Multiple images may belong to the same patient
- The dataset is highly imbalanced with very few positive samples
- Patient leakage can lead to overly optimistic evaluation results

To address these issues:
- Group-based splitting is used at the patient level
- Stratification is used to preserve target distribution

---

In [1]:
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit

RANDOM_STATE = 42

In [2]:
train_path = "../data/raw/train-metadata.csv"
df = pd.read_csv(train_path, low_memory=False)

print("Total samples:", len(df))
print("Unique patients:", df["patient_id"].nunique())
print("Positive samples:", df["target"].sum())
print("Negative samples:", len(df) - df["target"].sum())

Total samples: 401059
Unique patients: 1042
Positive samples: 393
Negative samples: 400666


### Dataset Overview

This dataset contains:
- Multiple samples from different patients
- A highly imbalanced target distribution
- Very few malignant (positive) cases compared to benign samples

Because of this:
- Random splitting is not appropriate
- Patient-level separation is required to avoid leakage

### Train / Temporary Split

The first split uses `StratifiedGroupKFold`.

This method combines:
- Stratification → preserves class distribution
- Grouping → ensures patients appear in only one split

The dataset is divided into:
- Train (~80%)
- Temporary (~20%)

The temporary set will later be divided into validation and holdout sets.

In [3]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

for fold, (train_idx, temp_idx) in enumerate(
    sgkf.split(
        df,
        y=df["target"],
        groups=df["patient_id"]
    )
):
    train_df = df.iloc[train_idx].reset_index(drop=True)
    temp_df = df.iloc[temp_idx].reset_index(drop=True)

    print(f"\nUsing Fold {fold}")
    break


Using Fold 0


In [4]:
print("\n===== SPLIT SIZE =====")
print("Train size:", len(train_df))
print("Temp size:", len(temp_df))


===== SPLIT SIZE =====
Train size: 320848
Temp size: 80211


### Initial Split Result

The training set contains most of the data for model learning.

The temporary split is reserved for evaluation purposes and will be separated into:
- Validation set
- Holdout set

### Validation / Holdout Split

The temporary set is further divided using `GroupShuffleSplit`.

This ensures:
- No patient overlap between splits
- More realistic evaluation performance
- Better simulation of real-world deployment

Final split ratio:
- Train: approximately 80%
- Validation: approximately 10%
- Holdout: approximately 10%

In [5]:
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.5,
    random_state=RANDOM_STATE
)

val_idx, holdout_idx = next(
    gss.split(
        temp_df,
        groups=temp_df["patient_id"]
    )
)

val_df = temp_df.iloc[val_idx].reset_index(drop=True)
holdout_df = temp_df.iloc[holdout_idx].reset_index(drop=True)

In [6]:
print("\n===== FINAL SPLIT SIZE =====")
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Holdout:", len(holdout_df))


===== FINAL SPLIT SIZE =====
Train: 320848
Validation: 38002
Holdout: 42209


### Final Split Summary

The dataset has now been separated into:
- Training set
- Validation set
- Holdout set

The holdout set remains completely unseen during model development.

### Patient Leakage Check

One of the most important requirements in medical machine learning is preventing patient leakage.

A patient must appear in only one split.

Otherwise:
- The model may memorize patient-specific patterns
- Evaluation scores may become artificially high

In [7]:
train_patients = set(train_df["patient_id"])
val_patients = set(val_df["patient_id"])
holdout_patients = set(holdout_df["patient_id"])

In [8]:
print("\n===== PATIENT OVERLAP CHECK =====")
print("Train ∩ Validation:", len(train_patients & val_patients))
print("Train ∩ Holdout:", len(train_patients & holdout_patients))
print("Validation ∩ Holdout:", len(val_patients & holdout_patients))


===== PATIENT OVERLAP CHECK =====
Train ∩ Validation: 0
Train ∩ Holdout: 0
Validation ∩ Holdout: 0


### Leakage Verification

All overlap counts should be zero.

This confirms:
- Complete patient-level separation
- No information leakage between splits

### Class Distribution Check

Because the dataset is highly imbalanced, it is important to verify that each split maintains a similar target distribution.

This helps ensure:
- Stable training
- Reliable validation
- Fair model evaluation

In [9]:
def check_distribution(name, dataframe):

    distribution = dataframe["target"].value_counts(normalize=True)

    print(f"\n{name} distribution:")
    print(distribution)

check_distribution("Train", train_df)
check_distribution("Validation", val_df)
check_distribution("Holdout", holdout_df)


Train distribution:
target
0    0.999018
1    0.000982
Name: proportion, dtype: float64

Validation distribution:
target
0    0.998816
1    0.001184
Name: proportion, dtype: float64

Holdout distribution:
target
0    0.999218
1    0.000782
Name: proportion, dtype: float64


### Distribution Verification

The positive / negative ratio remains relatively consistent across all splits.

This indicates that stratification was successful.

In [10]:
train_df.to_csv("../data/splits/train_split.csv", index=False)
val_df.to_csv("../data/splits/val_split.csv", index=False)
holdout_df.to_csv("../data/splits/holdout_split.csv", index=False)
print("\nSplits saved successfully!")


Splits saved successfully!


## Output Files

Generated files:
- `train_split.csv`
- `val_split.csv`
- `holdout_split.csv`

These files are stored in:
`../data/splits/`